In [1]:
class TwoTimeVF():
    '''
        autonomous velocity field: parameter mu

        containing: 
        velocity field: u, dim = 2
        Cartesian Hessian of each component: mat_D2u, list of matrix of dim 2
        Cartesian Gradient of u: mat_Du, matrix 2x2
    '''
    def __init__(self,mu,coef) -> None:
        r = sqrt(x**2+y**2)
        self.v_r = r*(1-r**2)
        self.v_t = mu - coef*y/r
        self.mu = mu
        self.coef = coef
        self.u = CF((self.v_r * x/r - y * self.v_t,
                    self.v_r * y/r + x * self.v_t))

        # derived Plane Hessian for each component of velocity
        self.mat_D2u = [
            CF((-(6*r*x - 2*self.coef*y**2/r**2 + 3*self.coef*y**4/r**4)/r, -y*(-self.coef*x/r + 3*self.coef*x**3/r**3 + 2*r**2)/r**2,
                -y*(-self.coef*x/r + 3*self.coef*x**3/r**3 + 2*r**2)/r**2, -x*(self.coef*x/r - 3*self.coef*x**3/r**3 + 2*r**2)/r**2), 
                dims=(2,2)),
            CF((-(-3*self.coef*x*y**3/r**4 + 2*r*y)/r, -x*(3*self.coef*x*y**2/r**3 + 2*r**2)/r**2, 
                -x*(3*self.coef*x*y**2/r**3 + 2*r**2)/r**2, -3*y*(-self.coef*x**3/r**3 + 2*r**2)/r**2), 
                dims=(2,2))
        ]
        # Plane Gradient -- Du: first row: D u_0, second row D u_1
        self.mat_Du = [
            [-self.coef*x*y**2/r**3 - 3*r**2 + 2*y**2 + 1, 2*self.coef*y/r - self.coef*y**3/r**3 - self.mu - 2*x*y],
            [-self.coef*y**3/r**3 + self.mu - 2*x*y,        -self.coef*x**3/r**3 - 3*r**2 + 2*x**2 + 1]
        ]
        self.mat_Du_CF = CF(
            tuple(self.mat_Du[0]+self.mat_Du[1]), dims=(2,2)
        )

In [2]:
import netgen.gui

optfile ./ng.opt does not exist - using default values
togl-version : 2
OCC module loaded
loading ngsolve library
NGSolve-6.2.2105
Using Lapack
Including sparse direct solver Pardiso
Running parallel using 8 thread(s)


In [8]:
from scipy.sparse import *
import numpy as np

def MyInv(Amat, Vec, FreeDofs:np.ndarray=None):
    '''
        # A is a ngsolve matrix，例如可以如下生成
        A = la.SparseMatrixd.CreateFromCOO([0,1,2], [0,1,2], [1,2,3], 3, 3)
        MyInv(A,BaseVector(np.array([1,2,3])),np.array([1,1,0],dtype=bool))
        gfu.vec.data += BaseVector(MyInv(a.mat,res:BaseVector,np.array(X.FreeDofs())))
        其中FreeDofs的dtype需要时bool才可以
    '''
    if FreeDofs is None:
        FreeDofs = np.array(np.ones(Vec.FV().NumPy().shape),dtype=bool)
    numFree = np.sum(FreeDofs)
    A_data = list(Amat.COO())
    
    A_coo = coo_matrix((A_data[2].NumPy(),(np.array(A_data[0]),np.array(A_data[1]))),Amat.shape)
    A_csr = A_coo.tocsr()
    A_new_csr = A_csr[FreeDofs][:, FreeDofs]
    b = Vec.FV().NumPy()[FreeDofs]
    
    # 使用spsolve求解Ax = b
    x = linalg.spsolve(A_new_csr, b)
    res = np.zeros(FreeDofs.shape)
    res[FreeDofs] = x
    return res

In [12]:
from ngsolve import *
import numpy as np
from netgen.geom2d import SplineGeometry

Output_option = True
dim = 2
tau0 = 2**(-7)
order = 2
print('spatial order is {}'.format(order))
        
tauval = tau0/2
T = 4
maxh = 0.05

tau = Parameter(tauval)
print(tauval)

Info_VF = TwoTimeVF(1.2,1)
x0,y0,r0 = 1/4,1/4,1/2
u = Info_VF.u

geo = SplineGeometry()
geo.AddCircle((x0,y0),r0,bc="circle")
mymesh = Mesh(geo.GenerateMesh(maxh=maxh))
mymesh.Curve(order)
fes = H1(mymesh,order=order)
fesV = VectorH1(mymesh,order=order)

told = 0
t_err_save = np.array([T])
err_set = []

Disp = GridFunction(fesV)
fesVD = VectorH1(mymesh,order=order,dirichlet=".*")
fesD = H1(mymesh,order=order,dirichlet=".*")
Lhs_H = BilinearForm(fesVD)
VExtend = GridFunction(fesVD)
VInterface = GridFunction(fesVD)
Rhs_H = LinearForm(fesVD)
v_d_trial, v_d_test = fesVD.TnT()
Lhs_H += -InnerProduct(grad(v_d_trial),grad(v_d_test))*dx
Rhs_H += InnerProduct(grad(VInterface),grad(v_d_test))*dx

t_sol = Parameter(0)
f = 1/(2+t_sol) + (4+4*x**2+4*y**2)*exp(x**2+y**2)
gfuInterface = GridFunction(fesD)
gfu = GridFunction(fesD)
utrial, utest = fesD.TnT()
uLhs = BilinearForm(fesD)
uRhs = LinearForm(fesD)
uLhs += 1/tau*utrial*utest*dx + InnerProduct(grad(utrial),grad(utest))*dx
uRhs += 1/tau*gfu*utest*dx + InnerProduct(VExtend,grad(gfu))*utest*dx + f*utest*dx\
    -  1/tau*gfuInterface*utest*dx - InnerProduct(grad(gfuInterface),grad(utest))*dx
errgfu = GridFunction(fesD)
gfuInterp = GridFunction(fesD)
Exactu = log(2+t_sol) - exp(x**2+y**2)
gfu.Set(Exactu) # set initial data

spatial order is 2
0.00390625
 Generate Mesh from spline geometry
 Curve elements, order = 2


In [13]:
SetVisualization(deformation=True)

In [14]:
Draw(Disp,mymesh,'disp',deformation=True)

In [15]:
import time

In [16]:
while told<=T:
    t_sol.Set(told)
    gfuInterface.Set(Exactu,definedon=mymesh.Boundaries('.*'))
    uLhs.Assemble()
    uRhs.Assemble()
    
    # gfu.vec.data = uLhs.mat.Inverse(freedofs=fesD.FreeDofs(),inverse='pardiso')*uRhs.vec + gfuInterface.vec
    gfu.vec.data = gfuInterface.vec + \
        BaseVector(MyInv(uLhs.mat,uRhs.vec,np.array(fesD.FreeDofs())))
    gfuInterp.Set(Exactu)
    errgfu.vec.data = gfuInterp.vec - gfu.vec

    tauval = tau.Get()
    VInterface.Interpolate(u,definedon=mymesh.Boundaries(".*"))
    Lhs_H.Assemble()
    Rhs_H.Assemble()
    # VExtend.vec.data = VInterface.vec + Lhs_H.mat.Inverse(freedofs=fesVD.FreeDofs(),inverse='pardiso')*Rhs_H.vec
    VExtend.vec.data = VInterface.vec.data + BaseVector(MyInv(Lhs_H.mat,Rhs_H.vec,np.array(fesVD.FreeDofs())))
    Disp.vec.data += BaseVector(tauval*VExtend.vec.FV().NumPy())
    mymesh.SetDeformation(Disp)
    Draw(gfu,mymesh,'sol')
    Redraw()
    time.sleep(0.01)
    told += tauval
    

Thank you for using NGSolve
